## Install Req

In [3]:
! pip install -q torch torchvision numpy matplotlib seaborn tqdm

You should consider upgrading via the '/Users/charvikusuma/Documents/Tarun/pc-experiments/venv/bin/python3 -m pip install --upgrade pip' command.


In [4]:
import os
import random
import numpy as np
from collections import namedtuple
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader, ConcatDataset, Subset

import matplotlib.pyplot as plt
import seaborn as sns



## Data Processing & Loaders

In [5]:
# EMNIST/Fashion are grayscale -> convert to 3 channels and 32x32
transform_gray_to_rgb = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.Grayscale(num_output_channels=3),  # 1ch -> 3ch
    transforms.ToTensor(),
])

# CIFAR already RGB 32x32
transform_cifar = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
])

# Datasets

train_set_1 = torchvision.datasets.EMNIST(
    root='./data',
    split='byclass',
    train=True,
    download=True,
    transform=transform_gray_to_rgb,
)

train_set_2 = torchvision.datasets.FashionMNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform_gray_to_rgb,
)

train_set_3 = torchvision.datasets.CIFAR100(
    root='./data',
    train=True,
    download=True,
    transform=transform_cifar,
)

# Label Remap
class OffsetLabels(Dataset):
    def __init__(self, base_ds, offset):
        self.base_ds = base_ds
        self.offset = offset
    def __len__(self):
        return len(self.base_ds)
    def __getitem__(self, idx):
        x, y = self.base_ds[idx]
        return x, y + self.offset

n1 = len(train_set_1.classes)  # EMNIST classes
n2 = len(train_set_2.classes)  # 10
n3 = len(train_set_3.classes)  # 100
train_all = ConcatDataset([
    OffsetLabels(train_set_1, 0),
    OffsetLabels(train_set_2, n1),
    OffsetLabels(train_set_3, n1 + n2),
])


num_total_classes = n1 + n2 + n3
print("Total classes:", num_total_classes)

Total classes: 172


In [6]:
def _targets_tensor(ds):
    # torchvision datasets usually expose labels in .targets
    if hasattr(ds, "targets"):
        t = ds.targets
    elif hasattr(ds, "train_labels"):  # fallback for older versions
        t = ds.train_labels
    else:
        raise ValueError(f"Could not find targets for dataset: {type(ds)}")

    if isinstance(t, list):
        t = torch.tensor(t)
    else:
        t = torch.as_tensor(t)
    return t.long()

def summarize_dataset(name, ds):
    y = _targets_tensor(ds)
    n_samples = len(ds)
    n_classes = len(ds.classes) if hasattr(ds, "classes") else int(y.max().item()) + 1
    counts = torch.bincount(y, minlength=n_classes)

    print(f"\n{name}")
    print(f"  samples: {n_samples}")
    print(f"  classes: {n_classes}")
    print("  per-class counts:")
    for i, c in enumerate(counts.tolist()):
        cls_name = ds.classes[i] if hasattr(ds, "classes") and i < len(ds.classes) else str(i)
        print(f"    class {i:>2} ({cls_name}): {c}")

summarize_dataset("EMNIST-byclass", train_set_1)
summarize_dataset("FashionMNIST", train_set_2)
summarize_dataset("CIFAR100", train_set_3)


EMNIST-byclass
  samples: 697932
  classes: 62
  per-class counts:
    class  0 (0): 34585
    class  1 (1): 38374
    class  2 (2): 34203
    class  3 (3): 35143
    class  4 (4): 33535
    class  5 (5): 31416
    class  6 (6): 34232
    class  7 (7): 35754
    class  8 (8): 33946
    class  9 (9): 33847
    class 10 (A): 6407
    class 11 (B): 3878
    class 12 (C): 10094
    class 13 (D): 4562
    class 14 (E): 4934
    class 15 (F): 9182
    class 16 (G): 2517
    class 17 (H): 3152
    class 18 (I): 11946
    class 19 (J): 3762
    class 20 (K): 2468
    class 21 (L): 5076
    class 22 (M): 9002
    class 23 (N): 8237
    class 24 (O): 24983
    class 25 (P): 8347
    class 26 (Q): 2605
    class 27 (R): 5073
    class 28 (S): 20764
    class 29 (T): 9820
    class 30 (U): 12602
    class 31 (V): 4637
    class 32 (W): 4695
    class 33 (X): 2771
    class 34 (Y): 4743
    class 35 (Z): 2701
    class 36 (a): 10033
    class 37 (b): 5159
    class 38 (c): 2854
    class 39 (d): 1

In [7]:
SEED = 42
BATCH_SIZE = 128
TEST_PER_CLASS = 50
PROTO_A_TRAIN_PER_CLASS = 450  # 500 total/class -> 450 train + 50 test

torch.manual_seed(SEED)

# train_set_1 = EMNIST, train_set_2 = FashionMNIST, train_set_3 = CIFAR100
datasets = [train_set_1, train_set_2, train_set_3]
n_classes = [len(d.classes) for d in datasets]
offsets = [0, n_classes[0], n_classes[0] + n_classes[1]]

def split_indices_per_class(ds, test_per_class=50):
    y = torch.as_tensor(ds.targets, dtype=torch.long)
    train_pool, test_idx = [], []
    for c in range(len(ds.classes)):
        idx = (y == c).nonzero(as_tuple=True)[0]
        idx = idx[torch.randperm(len(idx))]
        test_idx.extend(idx[:test_per_class].tolist())
        train_pool.append(idx[test_per_class:].tolist())
    return train_pool, test_idx

class OffsetSubset(Dataset):
    def __init__(self, ds, indices, offset):
        self.sub = Subset(ds, indices)
        self.offset = offset
    def __len__(self):
        return len(self.sub)
    def __getitem__(self, i):
        x, y = self.sub[i]
        return x, y + self.offset

# fixed test split (shared by Protocol A + B)
splits = [split_indices_per_class(ds, TEST_PER_CLASS) for ds in datasets]
test_sets = [OffsetSubset(ds, test_idx, off) for ds, off, (_, test_idx) in zip(datasets, offsets, splits)]
test_loader_all = DataLoader(ConcatDataset(test_sets), batch_size=BATCH_SIZE, shuffle=False)

# Protocol A: 450 train per class
protoA_sets = []
for ds, off, (pool, _) in zip(datasets, offsets, splits):
    idx = [i for cls_idx in pool for i in cls_idx[:PROTO_A_TRAIN_PER_CLASS]]
    protoA_sets.append(OffsetSubset(ds, idx, off))
protoA_loader = DataLoader(ConcatDataset(protoA_sets), batch_size=BATCH_SIZE, shuffle=True)

# Protocol B: all remaining train samples, sequential loaders
protoB_sets = []
for ds, off, (pool, _) in zip(datasets, offsets, splits):
    idx = [i for cls_idx in pool for i in cls_idx]
    protoB_sets.append(OffsetSubset(ds, idx, off))

protoB_seq_loaders = [DataLoader(s, batch_size=BATCH_SIZE, shuffle=True) for s in protoB_sets]

dataset_names = ["EMNIST", "FashionMNIST", "CIFAR100"]
test_loaders_per_ds = [DataLoader(ts, batch_size=BATCH_SIZE, shuffle=False) for ts in test_sets]

print("Protocol A train:", len(ConcatDataset(protoA_sets)))
print("Protocol B train sizes:", [len(s) for s in protoB_sets])
print("Shared test size:", len(ConcatDataset(test_sets)))

Protocol A train: 77400
Protocol B train sizes: [694832, 59500, 45000]
Shared test size: 8600


## Defining Networks

### PCN

In [8]:
# A simple container to keep error signals organized
# prediction = x_hat^(l)
# error = epsilon^(l)
# modulated_error = h^(l) this is fltered version of prediction error
LayerState = namedtuple('LayerState', ['prediction', 'error', 'modulated_error'])

class PCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, act_fn=torch.relu):
        super().__init__()
        # Weights translate "Concepts" from above into "Predictions" for below
        # Weights matrix: W^(l)
        self.W = nn.Parameter(torch.empty(out_dim, in_dim)) 
        nn.init.xavier_uniform_(self.W) 
        
        self.act_fn = act_fn
        # Derivative of activation function: f'(a)
        # For ReLU: 1 if a > 0, else 0
        self.act_deriv = lambda a: (a > 0).float() if act_fn == torch.relu else None

    def predict_down(self, x_above):
        """Higher layer tries to guess what happens in the layer below."""
        # Equation: a^(l) = x^(l+1) @ {W^(l)}^T
        # (Pre-activation prediction)
        pre_act = x_above @ self.W.T
        
        # Equation: x_hat^(l) = f(a^(l))
        # (Final top-down prediction)
        x_hat = self.act_fn(pre_act)
        return x_hat, pre_act

class PredictiveCodingNetwork(nn.Module):
    def __init__(self, dims, output_dim):  
        super().__init__()
        self.dims = dims
        self.L = len(dims) - 1
        
        # Build the hierarchy: [Input X0, Latent X1, Latent X2, ..., Top XL]
        self.layers = nn.ModuleList([
            PCNLayer(in_dim=dims[l+1], out_dim=dims[l])
            for l in range(self.L)
        ])
        
        # Build Readout Layer
        # Maps the top latent space directly to the class labels
        self.readout = nn.Linear(dims[-1], output_dim, bias=False)


    def init_latents(self, batch_size, device):
        """
        Creates the initial 'guesses' for every layer above the input.
        Returns: [X_1, X_2, ..., X_L]
        """
        # We start from index 1 because X_0 is provided by the actual data (the image)
        return [
           torch.randn(batch_size, d, device=device, requires_grad=False)
           for d in self.dims[1:]
        ]
        

    def compute_all_errors(self, latents):
            """
            latents: List of tensors [X_0, X_1, ..., X_L] 
            where X_0 is sensory data and X_L is the highest concept.
            Returns TWO lists: [errors], [modulated_errors]
            """
            errors = []
            gated_errors = []
            
            for layer, x_below, x_above in zip(self.layers, latents[:-1], latents[1:]):
                
                # 1. Prediction
                x_hat, pre_act = layer.predict_down(x_above)
                
                # 2. Raw Error
                error = x_below - x_hat
                
                # 3. Gain Modulation
                mod_error = error * layer.act_deriv(pre_act)
                
                # Append to separate lists for the training loop math
                errors.append(error)
                gated_errors.append(mod_error)
                
            return errors, gated_errors

### Resnet

In [ ]:
def make_resnet(num_classes):
    model = models.resnet18(num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model

resnet_model = make_resnet(num_total_classes)
print(f"ResNet Total params: {sum(p.numel() for p in resnet_model.parameters()):,}")

NameError: name 'models' is not defined

## Training Functions

In [ ]:
def train_pcn(model, data_loader, num_epochs, eta_infer, eta_learn, T_infer, T_learn, device='cuda'):
    """
    Trains a Predictive Coding Network.
    Unlike standard backprop, this operates in two phases per batch:
    1. Inference (Thinking): Update latent states (X) to minimize error.
    2. Learning (Memorizing): Update weights (W) using the settled latents.
    """
    model.to(device)
    model.train()

    # Trackers to see if the network is successfully "calming down" (minimizing energy)
    energy_history = []             
    supervised_energy_history = []
    accuracy_history = []

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1} / {num_epochs}")
        epoch_energies, epoch_supervised_energies, epoch_accuracies = [], [], []

        pbar = tqdm(data_loader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for x_batch, y_batch in pbar:   
            
            # PHASE 0: PREPARATION (The "Blank Slate")
            B = x_batch.size(0)
            
            # 1. Format Reality (Input & Output)
            x_batch = x_batch.view(B, -1).to(device)  # Flatten image (X_0)
            y_target = F.one_hot(y_batch, model.readout.out_features).float().to(device) # Target label (Y)
            
            # 2. Initialize the "Mind"
            # We create the stack of states: [Image(X0), RandomGuess(X1), RandomGuess(X2), ...]
            latents = [x_batch] + model.init_latents(B, device)
            
            # Extract weights for easy access: [W0, W1, ..., Readout_W]
            weights = [layer.W for layer in model.layers] + [model.readout.weight]

            batch_energies, batch_supervised_energies = [], []

            # PHASE 1: INFERENCE ("Thinking / Settling")
            # Goal: Nudge the latent guesses (X) until the predictions match reality.
            # We freeze the weights (no_grad) because we are changing *thoughts*, not *memory*.
            
            with torch.no_grad(), torch.autocast(device_type=device.type):
                
                # We iterate T_infer times to let the network reach a consensus
                for t in range(T_infer + 1): 
                    
                    # --- STEP A: What are our current errors? ---
                    # Calculate how wrong every layer's downward prediction is
                    errors, gated_errors = model.compute_all_errors(latents)
                    
                    # Calculate how wrong the TOP layer is about the classification label
                    y_hat = model.readout(latents[-1])  # Top concept tries to predict the label
                    eps_sup = y_hat - y_target          # Difference from true label
                    
                    # We project the label error back down to the top latent space
                    eps_L = eps_sup @ weights[-1]       
                    
                    # Combine all errors into one list: [Error_0, Error_1, ..., Error_Top]
                    errors_extended = errors + [eps_L]

                    # (Optional: Record the energy/error levels here for plotting)
                    sup_energy = 0.5 * eps_sup.pow(2).sum().item() / B
                    lat_energy = 0.5 * sum(e.pow(2).sum().item() for e in errors) / B
                    if t > 0: # Skip step 0 recording to align loops
                        batch_supervised_energies.append(sup_energy)
                        batch_energies.append(lat_energy + sup_energy)

                    # STEP B: Update the Latent States (X)
                    # We only update if we aren't on the very last step
                    if t < T_infer:
                        # Loop through every hidden layer (X_1 to X_L)
                        for l in range(1, model.L + 1):
                            
                            # THE CORE EQUATION OF PREDICTIVE CODING:
                            # 1. errors_extended[l]: The Boss is unhappy with me (Top-Down Error).
                            # 2. gated_errors[l-1] @ weights[l-1]: The Worker is unhappy with me (Bottom-Up Error).
                            # We combine these to find the perfect direction to nudge X.
                            
                            grad_Xl = errors_extended[l] - (gated_errors[l-1] @ weights[l-1])
                            
                            # Update the guess (Gradient Descent on the Energy)
                            latents[l] -= eta_infer * grad_Xl


            # PHASE 2: LEARNING ("Permanent Memory")
            # Goal: Now that the thoughts (X) have settled and errors are minimized,
            # we update the Weights (W) so the network makes better guesses *instantly* next time.
            
            with torch.no_grad(): # Standard PyTorch grad tracking is off; we do math manually
                
                # We iterate T_learn times (often just 1 time is enough in PCNs)
                for t in range(T_learn):
                    
                    # STEP A: Update the PCN Bridges (W_0 to W_{L-1}) 
                    for l in range(model.L):
                        
                        # Weight Gradient = -(Gated Error of Worker) * (State of Boss)
                        # "If the worker had an error, and the boss was active, change the connection."
                        grad_Wl = -(gated_errors[l].T @ latents[l+1]) / B
                        
                        # Update the weight matrix
                        weights[l] -= eta_learn * grad_Wl
                        
                    # STEP B: Update the Readout Classifier (W_out)
                    # Gradient = (Classification Error) * (Top Concept State)
                    grad_Wout = eps_sup.T @ latents[-1] / B
                    weights[-1] -= eta_learn * grad_Wout

                    # STEP C: Re-calculate errors for the next loop (if T_learn > 1)
                    errors, gated_errors = model.compute_all_errors(latents)
                    y_hat = model.readout(latents[-1])
                    eps_sup = y_hat - y_target


            batch_acc = (y_hat.argmax(dim=1) == y_batch.to(device)).float().mean().item()
            epoch_accuracies.append(batch_acc)

            epoch_energies.append(batch_energies)
            epoch_supervised_energies.append(batch_supervised_energies)

            pbar.set_postfix({
                "Energy": f"{batch_energies[-1]:.4f}",
                "Loss": f"{batch_supervised_energies[-1]:.4f}",
                "Acc": f"{batch_acc:.3f}"
            })

        # Save all batches for this epoch
        energy_history.append(epoch_energies)
        supervised_energy_history.append(epoch_supervised_energies)
        accuracy_history.append(epoch_accuracies)

    return energy_history, supervised_energy_history, accuracy_history

In [ ]:
def train_resnet(model, data_loader, num_epochs, device='cuda'):
    """Standard Feedforward Backprop Training"""
    model.to(device)
    model.train()

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0001)

    loss_history = []
    accuracy_history = []

    for epoch in range(num_epochs):
        pbar = tqdm(data_loader, desc=f"ResNet Epoch {epoch+1}/{num_epochs}")
        epoch_losses, epoch_accuracies = [], []

        for x_batch, y_batch in pbar:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()

            with torch.autocast(device_type=device.type if device.type != 'mps' else 'cpu'):
                logits = model(x_batch)
                loss = criterion(logits, y_batch)

            loss.backward()
            optimizer.step()

            batch_acc = (logits.argmax(dim=1) == y_batch).float().mean().item()
            epoch_losses.append(loss.item())
            epoch_accuracies.append(batch_acc)
            pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{batch_acc:.3f}"})

        loss_history.append(epoch_losses)
        accuracy_history.append(epoch_accuracies)

    return loss_history, accuracy_history

In [ ]:
# ==========================================
# DEVICE
# ==========================================
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

# ==========================================
# HYPERPARAMETERS
# ==========================================
input_dim = 3 * 32 * 32
output_dim = num_total_classes
layer_dims = [input_dim, 2048, 1024, 512, 256, 128]

num_epochs = 4
eta_infer = 0.05
eta_learn = 0.005
T_infer = 50
T_learn = 1

os.makedirs("checkpoints", exist_ok=True)

# ==========================================
# TEST FUNCTIONS
# ==========================================
@torch.no_grad()
def test_pcn(model, testloader, T_infer, eta_infer, device='cuda'):
    model.to(device).eval()
    total, top1, top3, loss_sum = 0, 0, 0, 0.0
    for x, y in tqdm(testloader, desc="PCN test", leave=False):
        B = x.size(0)
        total += B
        x = x.view(B, -1).to(device)
        y = y.to(device)
        latents = [x] + model.init_latents(B, device)
        weights = [layer.W for layer in model.layers] + [model.readout.weight]
        for _ in range(T_infer):
            errors, gated = model.compute_all_errors(latents)
            errs_ext = errors + [torch.zeros_like(latents[-1])]
            for l in range(1, model.L + 1):
                latents[l] -= eta_infer * (errs_ext[l] - gated[l-1] @ weights[l-1])
        logits = model.readout(latents[-1])
        loss_sum += F.cross_entropy(logits, y, reduction='sum').item()
        top1 += (logits.argmax(1) == y).sum().item()
        _, p3 = logits.topk(3, dim=1)
        top3 += (p3 == y.unsqueeze(1)).any(1).sum().item()
    model.train()
    return top1/total, top3/total, loss_sum/total

@torch.no_grad()
def test_resnet(model, testloader, device='cuda'):
    model.to(device).eval()
    total, top1, top3, loss_sum = 0, 0, 0, 0.0
    for x, y in tqdm(testloader, desc="ResNet test", leave=False):
        x, y = x.to(device), y.to(device)
        total += x.size(0)
        logits = model(x)
        loss_sum += F.cross_entropy(logits, y, reduction='sum').item()
        top1 += (logits.argmax(1) == y).sum().item()
        _, p3 = logits.topk(3, dim=1)
        top3 += (p3 == y.unsqueeze(1)).any(1).sum().item()
    model.train()
    return top1/total, top3/total, loss_sum/total

def run_full_test(pcn_model, resnet_model, device):
    """Test both models on per-dataset and combined test sets."""
    results = {}
    for name, loader in zip(dataset_names, test_loaders_per_ds):
        a1p, a3p, lp = test_pcn(pcn_model, loader, T_infer, eta_infer, device)
        a1r, a3r, lr_ = test_resnet(resnet_model, loader, device)
        results[name] = {
            'pcn_top1': a1p, 'pcn_top3': a3p, 'pcn_loss': lp,
            'resnet_top1': a1r, 'resnet_top3': a3r, 'resnet_loss': lr_,
        }
    a1p, a3p, lp = test_pcn(pcn_model, test_loader_all, T_infer, eta_infer, device)
    a1r, a3r, lr_ = test_resnet(resnet_model, test_loader_all, device)
    results['Combined'] = {
        'pcn_top1': a1p, 'pcn_top3': a3p, 'pcn_loss': lp,
        'resnet_top1': a1r, 'resnet_top3': a3r, 'resnet_loss': lr_,
    }
    for name, r in results.items():
        print(f"  {name:12s} | PCN: {r['pcn_top1']*100:.2f}% (loss={r['pcn_loss']:.3f}) | ResNet: {r['resnet_top1']*100:.2f}% (loss={r['resnet_loss']:.3f})")
    return results

In [ ]:
## Protocol A: Sample Efficiency
pcn_A = PredictiveCodingNetwork(dims=layer_dims, output_dim=output_dim)
resnet_A = make_resnet(output_dim)

print(f"PCN params:    {sum(p.numel() for p in pcn_A.parameters()):,}")
print(f"ResNet params: {sum(p.numel() for p in resnet_A.parameters()):,}")

print("\n--- PCN Protocol A ---")
pcn_A_energy, pcn_A_sup, pcn_A_acc = train_pcn(
    pcn_A, protoA_loader, num_epochs, eta_infer, eta_learn, T_infer, T_learn, device
)

print("\n--- ResNet Protocol A ---")
resnet_A_loss, resnet_A_acc = train_resnet(resnet_A, protoA_loader, num_epochs, device)

torch.save(pcn_A.state_dict(), "checkpoints/pcn_protoA.pt")
torch.save(resnet_A.state_dict(), "checkpoints/resnet_protoA.pt")
print("\nProtocol A checkpoints saved.")

In [ ]:
## Protocol B: Full Data Sequential
pcn_B = PredictiveCodingNetwork(dims=layer_dims, output_dim=output_dim)
resnet_B = make_resnet(output_dim)

pcn_B_train, resnet_B_train = {}, {}
pcn_B_forgetting, resnet_B_forgetting = [], []

for task_i, (name, loader) in enumerate(zip(dataset_names, protoB_seq_loaders)):
    print(f"\n{'='*50}")
    print(f"Task {task_i+1}/3: {name}")
    print(f"{'='*50}")

    e, se, acc = train_pcn(pcn_B, loader, num_epochs, eta_infer, eta_learn, T_infer, T_learn, device)
    pcn_B_train[name] = {'energy': e, 'sup_energy': se, 'accuracy': acc}

    loss_r, acc_r = train_resnet(resnet_B, loader, num_epochs, device)
    resnet_B_train[name] = {'loss': loss_r, 'accuracy': acc_r}

    pcn_row, resnet_row = [], []
    for t_name, t_loader in zip(dataset_names, test_loaders_per_ds):
        a1p, _, _ = test_pcn(pcn_B, t_loader, T_infer, eta_infer, device)
        a1r, _, _ = test_resnet(resnet_B, t_loader, device)
        pcn_row.append(a1p)
        resnet_row.append(a1r)
    pcn_B_forgetting.append(pcn_row)
    resnet_B_forgetting.append(resnet_row)

    print(f"\n  After {name}:")
    for t_name, pa, ra in zip(dataset_names, pcn_row, resnet_row):
        print(f"    {t_name}: PCN={pa*100:.1f}%, ResNet={ra*100:.1f}%")

torch.save(pcn_B.state_dict(), "checkpoints/pcn_protoB.pt")
torch.save(resnet_B.state_dict(), "checkpoints/resnet_protoB.pt")
print("\nProtocol B checkpoints saved.")

In [ ]:
## Evaluation

print("=== Protocol A ===")
protoA_test = run_full_test(pcn_A, resnet_A, device)

print("\n=== Protocol B (final) ===")
protoB_test = run_full_test(pcn_B, resnet_B, device)

## Visualization

In [ ]:
EARTH = {
    'terracotta': '#C86F54', 'sage': '#7A9B76', 'sand': '#C4A35A',
    'olive': '#5C6B3C', 'bg': '#FDFBF7', 'text': '#4E342E',
    'axes': '#5D4037', 'grid': '#EBE5D9',
}

def earthy():
    sns.set_theme(style="ticks", rc={
        "axes.facecolor": EARTH['bg'], "figure.facecolor": EARTH['bg'],
        "axes.edgecolor": EARTH['axes'], "text.color": EARTH['text'],
        "xtick.color": EARTH['text'], "ytick.color": EARTH['text'],
    })

def _annotate(ax, x, y, n=5, color=EARTH['text']):
    """Mark a few evenly-spaced points on a line."""
    if len(x) == 0:
        return
    idx = np.linspace(0, len(x)-1, min(n, len(x)), dtype=int)
    for i in idx:
        ax.annotate(f'{y[i]:.3f}', (x[i], y[i]), textcoords="offset points",
                    xytext=(0, 8), fontsize=7, ha='center', color=color, weight='bold')

def plot_training_curves(pcn_sup, pcn_acc, res_loss, res_acc, suffix=""):
    """Loss and Accuracy side-by-side: PCN vs ResNet."""
    earthy()
    pcn_l = [b[-1] for ep in pcn_sup for b in ep]
    pcn_a = [v for ep in pcn_acc for v in ep]
    res_l = [v for ep in res_loss for v in ep]
    res_a = [v for ep in res_acc for v in ep]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for ax, pd, rd, ylabel in [(ax1, pcn_l, res_l, "Loss"), (ax2, pcn_a, res_a, "Accuracy")]:
        xp, xr = np.arange(len(pd)), np.arange(len(rd))
        ax.plot(xp, pd, color=EARTH['terracotta'], lw=1.2, alpha=0.8, label='PCN')
        ax.plot(xr, rd, color=EARTH['sage'], lw=1.2, alpha=0.8, label='ResNet')
        _annotate(ax, xp, pd, 4, EARTH['terracotta'])
        _annotate(ax, xr, rd, 4, EARTH['sage'])
        ax.set_title(f"Training {ylabel}{suffix}", fontsize=13, weight='bold', pad=12)
        ax.set_xlabel("Batch", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.legend(frameon=False, fontsize=10)
        ax.grid(axis='y', ls='--', alpha=0.3, color=EARTH['grid'])
        sns.despine(ax=ax)

    plt.tight_layout()
    plt.savefig(f"checkpoints/train{suffix.replace(' ', '_').lower()}.png", dpi=150, bbox_inches='tight')
    plt.show()

def plot_test_bars(results, title="Test Accuracy"):
    """Grouped bars: PCN vs ResNet per dataset + combined."""
    earthy()
    labels = list(results.keys())
    pcn_v = [results[k]['pcn_top1'] * 100 for k in labels]
    res_v = [results[k]['resnet_top1'] * 100 for k in labels]
    x, w = np.arange(len(labels)), 0.3

    fig, ax = plt.subplots(figsize=(10, 5))
    b1 = ax.bar(x - w/2, pcn_v, w, color=EARTH['terracotta'], alpha=0.9, label='PCN')
    b2 = ax.bar(x + w/2, res_v, w, color=EARTH['sage'], alpha=0.9, label='ResNet')

    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            ax.annotate(f'{h:.1f}%', (bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 5), textcoords="offset points",
                        ha='center', fontsize=9, weight='bold', color=EARTH['text'])

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel("Top-1 Accuracy (%)", fontsize=11, weight='bold')
    ax.set_title(title, fontsize=14, weight='bold', pad=15)
    ax.set_ylim(0, max(max(pcn_v), max(res_v)) + 12)
    ax.legend(frameon=False, fontsize=10)
    ax.grid(axis='y', ls='--', alpha=0.3, color=EARTH['grid'])
    sns.despine(left=True, bottom=False)

    plt.tight_layout()
    plt.savefig(f"checkpoints/{title.replace(' ', '_').lower()}.png", dpi=150, bbox_inches='tight')
    plt.show()

def plot_sequential_training(pcn_train, res_train, names):
    """Protocol B: concatenated training curves with task boundary lines."""
    earthy()
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    pcn_l, res_l, pcn_a, res_a, bounds = [], [], [], [], [0]

    for nm in names:
        pl = [b[-1] for ep in pcn_train[nm]['sup_energy'] for b in ep]
        pa = [v for ep in pcn_train[nm]['accuracy'] for v in ep]
        rl = [v for ep in res_train[nm]['loss'] for v in ep]
        ra = [v for ep in res_train[nm]['accuracy'] for v in ep]
        pcn_l.extend(pl); pcn_a.extend(pa)
        res_l.extend(rl); res_a.extend(ra)
        bounds.append(len(pcn_l))

    for ax, pd, rd, ylabel in [(ax1, pcn_l, res_l, "Loss"), (ax2, pcn_a, res_a, "Accuracy")]:
        ax.plot(pd, color=EARTH['terracotta'], lw=1, alpha=0.8, label='PCN')
        ax.plot(rd, color=EARTH['sage'], lw=1, alpha=0.8, label='ResNet')
        for i, b in enumerate(bounds[1:-1]):
            ax.axvline(b, color=EARTH['axes'], ls=':', alpha=0.5)
            ax.text(b, ax.get_ylim()[1]*0.95, f' {names[i+1]}', fontsize=7,
                    color=EARTH['text'], va='top', ha='left', alpha=0.7)
        _annotate(ax, np.arange(len(pd)), pd, 5, EARTH['terracotta'])
        ax.set_title(f"Sequential {ylabel}", fontsize=13, weight='bold', pad=12)
        ax.set_xlabel("Batch", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.legend(frameon=False, fontsize=10)
        ax.grid(axis='y', ls='--', alpha=0.3, color=EARTH['grid'])
        sns.despine(ax=ax)

    plt.tight_layout()
    plt.savefig("checkpoints/train_protocol_b.png", dpi=150, bbox_inches='tight')
    plt.show()

def plot_forgetting_matrix(matrix, model_name, names):
    """Heatmap: accuracy on each dataset after each sequential task."""
    earthy()
    fig, ax = plt.subplots(figsize=(7, 5))
    data = np.array(matrix) * 100
    cmap = sns.color_palette([EARTH['bg'], EARTH['sand'], EARTH['terracotta']], as_cmap=True)
    sns.heatmap(data, annot=True, fmt='.1f', cmap=cmap,
                xticklabels=names,
                yticklabels=[f"After {n}" for n in names],
                ax=ax, cbar_kws={'label': 'Top-1 Acc (%)'},
                linewidths=0.5, linecolor=EARTH['grid'])
    ax.set_title(f"{model_name} — Forgetting Matrix", fontsize=13, weight='bold', pad=12)
    ax.set_xlabel("Tested On", fontsize=10)
    ax.set_ylabel("Trained Through", fontsize=10)
    plt.tight_layout()
    plt.savefig(f"checkpoints/forgetting_{model_name.lower()}.png", dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Protocol A
plot_training_curves(pcn_A_sup, pcn_A_acc, resnet_A_loss, resnet_A_acc, " (Protocol A)")
plot_test_bars(protoA_test, "Protocol A — Test Accuracy")

# Protocol B
plot_sequential_training(pcn_B_train, resnet_B_train, dataset_names)
plot_test_bars(protoB_test, "Protocol B — Test Accuracy")

# Forgetting matrices
plot_forgetting_matrix(pcn_B_forgetting, "PCN", dataset_names)
plot_forgetting_matrix(resnet_B_forgetting, "ResNet", dataset_names)